In [ ]:
import arcpy
import os
import csv

# =========================
# 0) 参数
# =========================
arcpy.env.overwriteOutput = True
scratch_gdb = arcpy.env.scratchGDB

# 省界（含 ADCODE99、NAME）
china_province = r"D:\path\to\sinkhole-risk-china\data\Administrative_divisions_of_china\china_pro2.shp"

# 县界（含 name、gb）
county_fc = r"D:\path\to\sinkhole-risk-china\data\Administrative_divisions_of_china\Administrative_divisions_of_china\county\县面.shp"

# 输出CSV（省名英文_分辨率）
province_en = "Guangdong"
resolution_km = 1
cell_size = 1000  # 3 km（米）

# ✅ 保存到 county 文件夹下
out_csv = rf"D:\path\to\sinkhole-risk-china\data\output\county\Points_{province_en}_{resolution_km}km.csv"


# 如果你已知广东 ADCODE99，可直接填（如 "440000"）；不填则脚本自动从 NAME 找“广东”
GUANGDONG_ADCODE99 = None

# =========================
# 1) 工具函数：判断是否经纬度（度）
# =========================
def is_degree_sr(spref: arcpy.SpatialReference) -> bool:
    try:
        return (spref.type == "Geographic") or (spref.linearUnitName is None) or ("Degree" in (spref.linearUnitName or ""))
    except:
        return True

def find_field_case_insensitive(fc, candidates):
    """在要素类里按不区分大小写找到字段名，返回实际字段名或 None"""
    fields = [f.name for f in arcpy.ListFields(fc)]
    low_map = {f.lower(): f for f in fields}
    for c in candidates:
        if c.lower() in low_map:
            return low_map[c.lower()]
    return None

# =========================
# 2) 检查必要字段是否存在
# =========================
prov_ad = find_field_case_insensitive(china_province, ["ADCODE99"])
prov_name = find_field_case_insensitive(china_province, ["NAME"])
if prov_ad is None or prov_name is None:
    raise RuntimeError("省界缺少字段 ADCODE99 或 NAME，请检查 china_pro2.shp 字段。")

county_name = find_field_case_insensitive(county_fc, ["name", "NAME"])
county_gb = find_field_case_insensitive(county_fc, ["gb", "GB"])
if county_name is None or county_gb is None:
    raise RuntimeError("县面缺少字段 name 或 gb，请检查 县面.shp 字段。")

# =========================
# 3) 找到广东 ADCODE99（若未手填）
# =========================
if GUANGDONG_ADCODE99 is None:
    gd_code = None
    with arcpy.da.SearchCursor(china_province, [prov_ad, prov_name]) as cur:
        for ad, nm in cur:
            if nm and ("广东" in str(nm)):
                gd_code = ad
                break
    if gd_code is None:
        raise RuntimeError("未在省界 NAME 字段中找到包含“广东”的记录。请手动设置 GUANGDONG_ADCODE99。")
    GUANGDONG_ADCODE99 = gd_code

print(f"广东 ADCODE99 = {GUANGDONG_ADCODE99}")

# =========================
# 4) 选择广东省面
# =========================
gd_layer = "gd_layer"
arcpy.management.MakeFeatureLayer(china_province, gd_layer)

ad_field_obj = [f for f in arcpy.ListFields(china_province) if f.name == prov_ad][0]
if ad_field_obj.type in ("String", "Guid"):
    where = f"{prov_ad} = '{GUANGDONG_ADCODE99}'"
else:
    where = f"{prov_ad} = {GUANGDONG_ADCODE99}"

arcpy.management.SelectLayerByAttribute(gd_layer, "NEW_SELECTION", where)

gd_poly = os.path.join(scratch_gdb, "Guangdong_polygon")
arcpy.management.CopyFeatures(gd_layer, gd_poly)

# =========================
# 5) 投影到米单位坐标系（用于3km网格）
# =========================
sr_in = arcpy.Describe(gd_poly).spatialReference

try:
    sr_grid = arcpy.SpatialReference("Asia North Albers Equal Area Conic")
except:
    sr_grid = arcpy.SpatialReference(3857)  # 兜底（单位米）

gd_for_grid = gd_poly
if is_degree_sr(sr_in):
    gd_for_grid = os.path.join(scratch_gdb, "Guangdong_projected_for_grid")
    arcpy.management.Project(gd_poly, gd_for_grid, sr_grid)
else:
    sr_grid = sr_in

# 县面也投影到同一坐标系，确保空间叠加正确
county_for_grid = county_fc
county_sr = arcpy.Describe(county_fc).spatialReference
if county_sr.name != sr_grid.name:
    county_for_grid = os.path.join(scratch_gdb, "County_projected_for_grid")
    arcpy.management.Project(county_fc, county_for_grid, sr_grid)

arcpy.env.outputCoordinateSystem = sr_grid
gd_extent = arcpy.Describe(gd_for_grid).extent

# =========================
# 6) 创建 3km 渔网 + 点必须落在面内（INSIDE）+ 裁剪到广东
# =========================
print("创建3km渔网并生成面内点（INSIDE）...")

fishnet = os.path.join(scratch_gdb, "fishnet_3km_poly")
origin_coord = f"{gd_extent.XMin} {gd_extent.YMin}"
y_axis_coord = f"{gd_extent.XMin} {gd_extent.YMax}"
corner_coord = f"{gd_extent.XMax} {gd_extent.YMax}"

arcpy.management.CreateFishnet(
    out_feature_class=fishnet,
    origin_coord=origin_coord,
    y_axis_coord=y_axis_coord,
    cell_width=cell_size,
    cell_height=cell_size,
    number_rows=0,
    number_columns=0,
    corner_coord=corner_coord,
    labels="NO_LABELS",
    template="#",
    geometry_type="POLYGON"
)

# ✅ 点必须落在面内：INSIDE
fishnet_points = os.path.join(scratch_gdb, "fishnet_3km_points_inside_cell")
arcpy.management.FeatureToPoint(fishnet, fishnet_points, "INSIDE")

# ✅ 裁剪到广东：确保点在广东省面内
gd_points_proj = os.path.join(scratch_gdb, "Guangdong_3km_points_projected")
arcpy.analysis.Clip(fishnet_points, gd_for_grid, gd_points_proj)

# =========================
# 7) Spatial Join：先加省属性（ADCODE99），再加县属性（name/gb）
# =========================
print("叠加省属性与县属性...")

# 7.1 省属性
gd_points_with_prov = os.path.join(scratch_gdb, "Guangdong_3km_points_with_prov")
arcpy.analysis.SpatialJoin(
    target_features=gd_points_proj,
    join_features=gd_for_grid,
    out_feature_class=gd_points_with_prov,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    match_option="INTERSECT"
)

# 7.2 县属性
gd_points_with_county = os.path.join(scratch_gdb, "Guangdong_3km_points_with_county")
arcpy.analysis.SpatialJoin(
    target_features=gd_points_with_prov,
    join_features=county_for_grid,
    out_feature_class=gd_points_with_county,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    match_option="INTERSECT"
)

# =========================
# 8) 投影到WGS84并添加字段 Province / Longitude / Latitude
# =========================
print("投影到WGS84并计算经纬度...")
wgs84 = arcpy.SpatialReference(4326)
gd_points_wgs84 = os.path.join(scratch_gdb, "Guangdong_3km_points_WGS84")
arcpy.management.Project(gd_points_with_county, gd_points_wgs84, wgs84)

# 添加字段 Province / Longitude / Latitude
fnames = [f.name for f in arcpy.ListFields(gd_points_wgs84)]
if "Province" not in fnames:
    arcpy.management.AddField(gd_points_wgs84, "Province", "TEXT", field_length=100)
if "Longitude" not in fnames:
    arcpy.management.AddField(gd_points_wgs84, "Longitude", "DOUBLE")
if "Latitude" not in fnames:
    arcpy.management.AddField(gd_points_wgs84, "Latitude", "DOUBLE")

with arcpy.da.UpdateCursor(gd_points_wgs84, ["SHAPE@X", "SHAPE@Y", "Province", "Longitude", "Latitude"]) as cur:
    for x, y, prov, lon, lat in cur:
        cur.updateRow((x, y, province_en, x, y))

# =========================
# 9) 识别 join 后的字段名（可能带 _1）
# =========================
fnames = [f.name for f in arcpy.ListFields(gd_points_wgs84)]

# 省ADC字段
ad_out = "ADCODE99" if "ADCODE99" in fnames else ("ADCODE99_1" if "ADCODE99_1" in fnames else None)
if ad_out is None:
    # 有时候会保留原字段名（prov_ad），再兜底一次
    ad_out = prov_ad if prov_ad in fnames else None
if ad_out is None:
    raise RuntimeError("输出点中找不到 ADCODE99 字段（或其 _1 版本），请检查 Spatial Join 输出。")

# 县字段
cname_out = county_name if county_name in fnames else (f"{county_name}_1" if f"{county_name}_1" in fnames else None)
cgb_out = county_gb if county_gb in fnames else (f"{county_gb}_1" if f"{county_gb}_1" in fnames else None)
if cname_out is None or cgb_out is None:
    raise RuntimeError("输出点中找不到县域字段 name/gb（或其 _1 版本），请检查 Spatial Join 输出。")

# =========================
# 10) 导出 CSV：No, ADCODE99, Province, County, gb, Longitude, Latitude
# =========================
print("导出CSV...")
out_dir = os.path.dirname(out_csv)
os.makedirs(out_dir, exist_ok=True)

rows = 0
with open(out_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["No", "ADCODE99", "Province", "County", "gb", "Longitude", "Latitude"])

    with arcpy.da.SearchCursor(
        gd_points_wgs84,
        [ad_out, "Province", cname_out, cgb_out, "Longitude", "Latitude"]
    ) as cur:
        i = 1
        for ad, prov, cname, cgb, lon, lat in cur:
            writer.writerow([i, ad, prov, cname, cgb, lon, lat])
            i += 1
            rows += 1

print(f"完成！广东 {resolution_km}km 网格点数量：{rows}")
print(f"CSV 已保存：{out_csv}")


下面是一次性生成34个区域的代码

In [ ]:
import arcpy
import os
import csv
import re

# =========================
# 0) 参数
# =========================
arcpy.env.overwriteOutput = True
scratch_gdb = arcpy.env.scratchGDB

china_province = r"D:\path\to\sinkhole-risk-china\data\Administrative_divisions_of_china\china_pro2.shp"
county_fc = r"D:\path\to\sinkhole-risk-china\data\Administrative_divisions_of_china\Administrative_divisions_of_china\county\县面.shp"

resolution_km = 1
cell_size = 1000  # 3km（米）

out_dir = r"D:\path\to\sinkhole-risk-china\data\output\county"

# =========================
# 1) 省名中文 -> 英文（严格用于文件名/Province字段）
# =========================
PROVINCE_EN_MAP = {
    "北京": "Beijing",
    "天津": "Tianjin",
    "河北": "Hebei",
    "山西": "Shanxi",
    "内蒙古": "InnerMongolia",
    "辽宁": "Liaoning",
    "吉林": "Jilin",
    "黑龙江": "Heilongjiang",
    "上海": "Shanghai",
    "江苏": "Jiangsu",
    "浙江": "Zhejiang",
    "安徽": "Anhui",
    "福建": "Fujian",
    "江西": "Jiangxi",
    "山东": "Shandong",
    "河南": "Henan",
    "湖北": "Hubei",
    "湖南": "Hunan",
    "广东": "Guangdong",
    "广西": "Guangxi",
    "海南": "Hainan",
    "重庆": "Chongqing",
    "四川": "Sichuan",
    "贵州": "Guizhou",
    "云南": "Yunnan",
    "西藏": "Tibet",
    "陕西": "Shaanxi",
    "甘肃": "Gansu",
    "青海": "Qinghai",
    "宁夏": "Ningxia",
    "新疆": "Xinjiang",
    "香港": "HongKong",
    "澳门": "Macau",
    "台湾": "Taiwan",
}

# =========================
# 2) 工具函数
# =========================
def is_degree_sr(spref: arcpy.SpatialReference) -> bool:
    try:
        return (spref.type == "Geographic") or (spref.linearUnitName is None) or ("Degree" in (spref.linearUnitName or ""))
    except:
        return True

def find_field_case_insensitive(fc, candidates):
    fields = [f.name for f in arcpy.ListFields(fc)]
    low_map = {f.lower(): f for f in fields}
    for c in candidates:
        if c.lower() in low_map:
            return low_map[c.lower()]
    return None

def safe_filename(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r'[\\/:*?"<>|]+', "_", name)
    name = re.sub(r"\s+", "_", name)
    return name

def normalize_province_cn(name_cn: str) -> str:
    """
    将 NAME 中的省名做归一化，尽量得到 dict 的 key（如：广东/内蒙古/新疆/香港/澳门/台湾）。
    兼容：省/市/自治区/特别行政区/壮族/回族/维吾尔 等后缀。
    """
    if name_cn is None:
        return ""
    s = str(name_cn).strip()
    # 常见后缀/词去除
    for k in ["省", "市", "自治区", "特别行政区", "壮族", "回族", "维吾尔", "维吾尔族"]:
        s = s.replace(k, "")
    s = s.replace("新疆维吾尔", "新疆")  # 双保险
    s = s.replace("内蒙古", "内蒙古")
    s = s.replace("广西", "广西")
    s = s.replace("宁夏", "宁夏")
    s = s.replace("西藏", "西藏")
    return s

def province_cn_to_en(name_cn: str):
    """
    返回英文省名；如果 name_cn 为空则返回 None（让上层跳过）。
    """
    if name_cn is None:
        return None
    s = str(name_cn).strip()
    if s == "":
        return None

    key = normalize_province_cn(s)
    if key in PROVINCE_EN_MAP:
        return PROVINCE_EN_MAP[key]

    raise RuntimeError(f"未能将省名映射为英文：NAME='{name_cn}' 归一化后='{key}'。请补充 PROVINCE_EN_MAP 或调整归一化规则。")

# =========================
# 3) 字段检查
# =========================
prov_ad = find_field_case_insensitive(china_province, ["ADCODE99"])
prov_name = find_field_case_insensitive(china_province, ["NAME"])
if prov_ad is None or prov_name is None:
    raise RuntimeError("省界缺少字段 ADCODE99 或 NAME，请检查 china_pro2.shp。")

county_name = find_field_case_insensitive(county_fc, ["name", "NAME"])
county_gb = find_field_case_insensitive(county_fc, ["gb", "GB"])
if county_name is None or county_gb is None:
    raise RuntimeError("县面缺少字段 name 或 gb，请检查 县面.shp。")

# =========================
# 4) 统一投影坐标系（用于生成3km网格）
# =========================
try:
    sr_grid_default = arcpy.SpatialReference("Asia North Albers Equal Area Conic")
except:
    sr_grid_default = arcpy.SpatialReference(3857)

sr_prov_in = arcpy.Describe(china_province).spatialReference
china_prov_for_grid = china_province
sr_grid = sr_prov_in

if is_degree_sr(sr_prov_in):
    sr_grid = sr_grid_default
    china_prov_for_grid = os.path.join(scratch_gdb, "china_province_projected_for_grid")
    arcpy.management.Project(china_province, china_prov_for_grid, sr_grid)

county_for_grid = county_fc
sr_county_in = arcpy.Describe(county_fc).spatialReference
if sr_county_in.name != sr_grid.name:
    county_for_grid = os.path.join(scratch_gdb, "county_projected_for_grid")
    arcpy.management.Project(county_fc, county_for_grid, sr_grid)

arcpy.env.outputCoordinateSystem = sr_grid
os.makedirs(out_dir, exist_ok=True)

# =========================
# 5) 汇总省记录（按 ADCODE99 去重）
# =========================
province_records = {}
with arcpy.da.SearchCursor(china_prov_for_grid, [prov_ad, prov_name]) as cur:
    for ad, name_cn in cur:
        if ad not in province_records:
            province_records[ad] = name_cn

print(f"将处理省级单元数量：{len(province_records)}")

# =========================
# 6) 遍历每个省：生成 3km INSIDE 点 + 叠加县域 + 输出 CSV
# =========================
wgs84 = arcpy.SpatialReference(4326)

for idx, (adcode, name_cn) in enumerate(province_records.items(), start=1):
    prov_en = province_cn_to_en(name_cn)   # ✅ 严格英文
    prov_en_safe = safe_filename(prov_en)

    out_csv = os.path.join(out_dir, f"Points_{prov_en_safe}_{resolution_km}km.csv")
    print(f"\n[{idx}/{len(province_records)}] 处理：{name_cn} -> {prov_en} (ADCODE99={adcode})")
    print(f"输出：{out_csv}")

    # 6.1 选择该省面
    lyr = "prov_lyr"
    arcpy.management.MakeFeatureLayer(china_prov_for_grid, lyr)

    ad_field_obj = [f for f in arcpy.ListFields(china_prov_for_grid) if f.name == prov_ad][0]
    if ad_field_obj.type in ("String", "Guid"):
        where = f"{prov_ad} = '{adcode}'"
    else:
        where = f"{prov_ad} = {adcode}"

    arcpy.management.SelectLayerByAttribute(lyr, "NEW_SELECTION", where)

    prov_poly = os.path.join(scratch_gdb, f"prov_poly_{idx}")
    arcpy.management.CopyFeatures(lyr, prov_poly)

    # 6.2 创建 3km 渔网 + 点必须落在面内（INSIDE）+ Clip 到省面
    ext = arcpy.Describe(prov_poly).extent

    fishnet = os.path.join(scratch_gdb, f"fishnet_{idx}")
    origin_coord = f"{ext.XMin} {ext.YMin}"
    y_axis_coord = f"{ext.XMin} {ext.YMax}"
    corner_coord = f"{ext.XMax} {ext.YMax}"

    arcpy.management.CreateFishnet(
        out_feature_class=fishnet,
        origin_coord=origin_coord,
        y_axis_coord=y_axis_coord,
        cell_width=cell_size,
        cell_height=cell_size,
        number_rows=0,
        number_columns=0,
        corner_coord=corner_coord,
        labels="NO_LABELS",
        template="#",
        geometry_type="POLYGON"
    )

    fishnet_points = os.path.join(scratch_gdb, f"fishnet_pts_{idx}")
    arcpy.management.FeatureToPoint(fishnet, fishnet_points, "INSIDE")  # ✅ 必须落在面内

    prov_points_proj = os.path.join(scratch_gdb, f"prov_pts_clip_{idx}")
    arcpy.analysis.Clip(fishnet_points, prov_poly, prov_points_proj)

    # 6.3 Spatial Join：省属性 + 县属性
    pts_with_prov = os.path.join(scratch_gdb, f"pts_with_prov_{idx}")
    arcpy.analysis.SpatialJoin(
        target_features=prov_points_proj,
        join_features=prov_poly,
        out_feature_class=pts_with_prov,
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_COMMON",
        match_option="INTERSECT"
    )

    pts_with_county = os.path.join(scratch_gdb, f"pts_with_county_{idx}")
    arcpy.analysis.SpatialJoin(
        target_features=pts_with_prov,
        join_features=county_for_grid,
        out_feature_class=pts_with_county,
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_COMMON",
        match_option="INTERSECT"
    )

    # 6.4 投影到 WGS84 + 计算经纬度 + 写 Province(英文)
    pts_wgs84 = os.path.join(scratch_gdb, f"pts_wgs84_{idx}")
    arcpy.management.Project(pts_with_county, pts_wgs84, wgs84)

    fnames = [f.name for f in arcpy.ListFields(pts_wgs84)]
    if "Province" not in fnames:
        arcpy.management.AddField(pts_wgs84, "Province", "TEXT", field_length=100)
    if "Longitude" not in fnames:
        arcpy.management.AddField(pts_wgs84, "Longitude", "DOUBLE")
    if "Latitude" not in fnames:
        arcpy.management.AddField(pts_wgs84, "Latitude", "DOUBLE")

    with arcpy.da.UpdateCursor(pts_wgs84, ["SHAPE@X", "SHAPE@Y", "Province", "Longitude", "Latitude"]) as cur2:
        for x, y, provv, lon, lat in cur2:
            cur2.updateRow((x, y, prov_en, x, y))

    # 6.5 识别 join 后字段名（可能 _1）
    fnames = [f.name for f in arcpy.ListFields(pts_wgs84)]

    ad_out = prov_ad if prov_ad in fnames else (f"{prov_ad}_1" if f"{prov_ad}_1" in fnames else None)
    if ad_out is None:
        ad_out = "ADCODE99" if "ADCODE99" in fnames else ("ADCODE99_1" if "ADCODE99_1" in fnames else None)
    if ad_out is None:
        raise RuntimeError(f"[{prov_en}] 输出点中找不到 ADCODE99 字段（或 _1）。")

    cname_out = county_name if county_name in fnames else (f"{county_name}_1" if f"{county_name}_1" in fnames else None)
    cgb_out = county_gb if county_gb in fnames else (f"{county_gb}_1" if f"{county_gb}_1" in fnames else None)
    if cname_out is None or cgb_out is None:
        raise RuntimeError(f"[{prov_en}] 输出点中找不到县域字段 name/gb（或 _1）。")

    # 6.6 输出 CSV
    with open(out_csv, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["No", "ADCODE99", "Province", "County", "gb", "Longitude", "Latitude"])

        i = 1
        with arcpy.da.SearchCursor(pts_wgs84, [ad_out, "Province", cname_out, cgb_out, "Longitude", "Latitude"]) as cur3:
            for ad_v, prov_v, county_v, gb_v, lon_v, lat_v in cur3:
                writer.writerow([i, ad_v, prov_v, county_v, gb_v, lon_v, lat_v])
                i += 1

    npts = int(arcpy.management.GetCount(pts_wgs84)[0])
    print(f"完成：{prov_en} -> 点数 {npts}")

print("\n全部省份处理完成！")
